# Clase 8 — Whisper: transcribiendo voz a texto

## Objetivos de aprendizaje
- Explicar qué es Whisper y por qué es un sistema de reconocimiento de voz (ASR) robusto.
- Distinguir los tamaños de modelo de Whisper y su relación precisión/recursos.
- Usar Whisper vía `transformers.pipeline` para transcribir un audio real.
- Reconocer los casos de uso reales de Whisper (subtitulado, transcripción de reuniones, etc).

## Qué vamos a construir
Vamos a descargar un audio de ejemplo público y transcribirlo con Whisper
(`openai/whisper-tiny.en`) usando el pipeline de Hugging Face —la misma vía que usa el
proyecto real "AI Meeting Companion" de IBM—, sin necesidad de ninguna clave de API.

## Whisper, en corto

Whisper es un sistema de reconocimiento automático del habla (ASR, *Automatic Speech
Recognition*) de OpenAI, entrenado por deep learning sobre una cantidad enorme de audio real.
Sus tres fortalezas clave:
- **Alta precisión**, incluso con audio de baja calidad.
- **Soporte multilingüe** automático (no hace falta indicarle el idioma).
- **Robustez frente al ruido** de fondo.

Existen 5 tamaños de modelo: `tiny`, `base`, `small`, `medium`, `large` — a mayor tamaño,
mayor precisión pero más recursos necesarios. Casos de uso reales: subtitulado automático de
videos, buscadores por voz, transcripción de reuniones, soporte al cliente multilingüe.

Repasa la explicación completa y la analogía en el
[Capítulo 12 del manual](../manual/12_whisper_voz.md).

In [ ]:
# transformers y torch suelen venir preinstalados en Colab; los instalamos
# igualmente para asegurar versiones compatibles con el pipeline de ASR
!pip install -q transformers torch requests

In [ ]:
import requests

# Audio de ejemplo público usado en el proyecto real "AI Meeting Companion" de IBM
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-GPXX04C6EN/Testing%20speech%20to%20text.mp3"

response = requests.get(url)
audio_file_path = "downloaded_audio.mp3"

if response.status_code == 200:
    with open(audio_file_path, "wb") as file:
        file.write(response.content)
    print("Audio descargado correctamente.")
else:
    print("No se pudo descargar el audio. Código:", response.status_code)

In [ ]:
from transformers import pipeline

# Igual que en el proyecto real AIMeetingCompanion_Project/simple_speech2text.py:
# usamos Whisper vía transformers.pipeline en vez de la librería openai-whisper
# directa, porque es más liviano de instalar en un entorno gratuito como Colab.
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny.en",
    chunk_length_s=30,
)

# batch_size=8 indica cuántos trozos de audio se procesan a la vez
prediction = pipe(audio_file_path, batch_size=8)["text"]
print("Transcripción:")
print(prediction)

## Referencia: la librería oficial `openai-whisper`

El PDF fuente de OpenAI Whisper documenta este código real (también gratuito, pero más
pesado de instalar en Colab):

```bash
pip install git+https://github.com/openai/whisper.git
```

```python
import whisper

model = whisper.load_model("base")
result = model.transcribe("audio_example.mp3")
print(result["text"])
```

## Referencia: Whisper integrado en un servidor Flask

Retomando lo que viste en la clase 7, así se integraría la transcripción en un endpoint
Flask:

```python
from flask import Flask, request
import whisper

app = Flask(__name__)
model = whisper.load_model("base")

@app.route("/transcribe", methods=["POST"])
def transcribe_audio():
    audio_file = request.files["audio"]
    result = model.transcribe(audio_file)
    return {"transcription": result["text"]}

if __name__ == "__main__":
    app.run(debug=True)
```

## Del audio al resumen: el caso de uso completo

El proyecto real "AI Meeting Companion" no se queda en la transcripción: le pasa el texto
transcrito a un LLM pidiéndole "los puntos clave de la reunión". En el material original de
IBM ese paso lo hace un modelo de pago de watsonx. En la práctica de esta clase vas a resolver
ese mismo paso con un modelo de resumen gratuito de Hugging Face.

## Resumen de la clase
- Whisper es un sistema ASR robusto, multilingüe y preciso incluso con audio de baja calidad.
- Existen 5 tamaños de modelo (`tiny` a `large`); usamos `whisper-tiny.en` por ser el más liviano.
- `transformers.pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")` es la vía más práctica para usar Whisper en Colab, sin instalar la librería oficial.
- El caso de uso real de este bloque del curso es transcribir + resumir reuniones.

Continúa con la [práctica de esta clase](../notebooks_practica/clase08_practica_whisper_voz.ipynb)
y repasa el [Capítulo 12 del manual](../manual/12_whisper_voz.md).